# DREAMwalk MSI: 10 x 10 cross-validation with fixed embeddings

Compare **Original** and **Modified** DREAMwalk on the Figure 2a random-pair task using the full MSI label cohort. Select the existing **DREAMwalk Python 3.8 kernel** and run the notebook in order or choose **Run All**.

- Ten repeats use seeds 0-9; each repeat has ten stratified folds.
- Each repeat has all 5,926 positives and 5,926 unknown pairs sampled as negatives. Negative sets do not overlap across repeats.
- Both variants use identical pair IDs and folds. Only their saved embedding vectors differ.
- The two MeSH2024, seed-0 embeddings stay fixed: tau=0.3, pp=1, cutoff=0.4. No graph, walks or embedding training is repeated.
- Section 6 imports the existing ten-fold seed-0 results after checking inputs and settings. Sections **7 and 8** start the **180 new classifier fits**. Each fold is saved immediately, so rerunning resumes completed work.
- There are 200 fold results in total: 100 per model, including 20 imported seed-0 results.

[Bang et al. (2023)](https://doi.org/10.1038/s41467-023-39301-y), Methods pages 13-14, specifies ten-times ten-fold CV with disjoint negative sets. Source Data, sheet `Figure2a`, labels its ten DREAMwalk records with seeds 0-9. It does not supply individual fold results or specify whether embeddings were retrained for every Figure 2a repeat. Report this notebook as **10 x 10 CV with fixed embeddings**. Tau 0.3 is the accepted local setting, not a claimed explicitly published Figure 2a value.

The disease-area evaluation remains separate. This notebook uses all 840 diseases, including those without a usable MeSH category.


## 1. Paths and existing environment

Interpreter relative to the repository: `00.Reproduction/00.DREAMwalk_reproduction/environment/.conda-env/bin/python`. If the notebook starts outside the repository, set `PROJECT_OVERRIDE` to the DREAMwalk project directory.


In [3]:
from pathlib import Path
import sys
import json
import hashlib
from importlib.metadata import version

PROJECT_OVERRIDE = None
start = Path(PROJECT_OVERRIDE).expanduser().resolve() if PROJECT_OVERRIDE else Path.cwd().resolve()
ROOT = next((candidate
             for parent in [start, *start.parents]
             for candidate in [parent, parent/'00.DREAMwalk_reproduction',
                               parent/'00.Reproduction/00.DREAMwalk_reproduction']
             if (candidate/'src/msi_repeated_cv.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Set PROJECT_OVERRIDE to the DREAMwalk project directory.')
if sys.version_info[:2] != (3, 8):
    raise RuntimeError('Select the existing DREAMwalk Python 3.8 kernel: ' +
                       str(ROOT/'environment/.conda-env/bin/python'))
sys.path.insert(0, str(ROOT/'src'))
import numpy as np
import pandas as pd
from IPython.display import display
import msi_repeated_cv as cv
from run_cv import inputs
from run_msi_repeat0 import xgb_settings, sha256

print('Project:', ROOT)
print('Interpreter:', sys.executable)
print('Packages:', {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'scipy', 'xgboost']})


Project: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction
Interpreter: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/environment/.conda-env/bin/python
Packages: {'numpy': '1.23.3', 'pandas': '1.4.4', 'scikit-learn': '1.1.3', 'scipy': '1.9.1', 'xgboost': '1.7.4'}


## 2. Fixed settings and the two model variants

Original retains the released teleport selector and input-type pp=1 NS. Modified uses first-hit teleport, positive-output-type NS and current-positive rejection. Both use the same reviewed MeSH2024 semantic input.

The repeat seed controls Link-level negative sampling, fold allocation and the XGBoost seed. Other XGBoost parameters remain unchanged: 500 trees, maximum depth 6 and four workers. Fold numbers 1-10 are indices, not a second set of seeds. No hyperparameter search or early stopping is used.


In [4]:
RUN_NAME = 'msi_mesh2024_fixed_embeddings_10x10'
RESULTS = ROOT/'results'/RUN_NAME
REPEAT_SEEDS = list(range(10))
N_FOLDS = 10
XGB_PARAMETERS = xgb_settings()
MODELS = {
    'original': 'msi_seed0_semantic_pp_mesh2024_roots',
    'modified': 'msi_seed0_mesh2024_final_fixed',
}
EMBEDDINGS = {name: ROOT/'data/embeddings'/run/'pp1.pkl' for name, run in MODELS.items()}
HISTORICAL = {name: ROOT/'results'/run for name, run in MODELS.items()}
PAPER_REFERENCE = ROOT/'references/msi_fig2a_dreamwalk.csv'
SEED0_NEGATIVES = ROOT/'results/msi_repeat0_negative_pairs.csv'
for name in MODELS:
    if not EMBEDDINGS[name].is_file():
        raise FileNotFoundError(EMBEDDINGS[name])
    source = json.loads((HISTORICAL[name]/'run_config.json').read_text())
    assert source['semantic'] == {'mesh_year': 2024, 'cutoff': 0.4, 'letter_roots': True}
    assert source['walks'] == dict(tp_factor=0.3, seed=0, num_walks=100,
                                  walk_length=10, p=1, q=1, workers=4, em_max_iter=5)
    assert source['xgboost'] == XGB_PARAMETERS
    assert source['folds'] == N_FOLDS and source['cv_seed'] == 0
    print(name, 'fixed embedding:', EMBEDDINGS[name])
print('Repeat seeds:', REPEAT_SEEDS, '| folds per repeat:', N_FOLDS)
display(pd.DataFrame([XGB_PARAMETERS]))


original fixed embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/pp1.pkl
modified fixed embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/pp1.pkl
Repeat seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | folds per repeat: 10


,base_score,booster,eval_metric,objective,gamma,learning_rate,max_depth,n_estimators,tree_method,min_child_weight,subsample,colsample_bytree,scale_pos_weight,max_delta_step,seed,n_jobs
0,0.5,gbtree,error,binary:logistic,0,0.1,6,500,auto,4,0.8,0.9,1,1,0,4


## 3. Full MSI cohort and ten disjoint negative sets

Use 1,661 drugs, 840 diseases and all 5,926 positive pairs. No disease-category filtering is applied. Each repeat contains 11,852 labelled pairs.

Seed 0 preserves the existing negative-pair order. For each later seed, sample uniformly without replacement from the sorted drug-by-disease candidates after excluding known positives and earlier negative sets. Thus, the ten sets contain 59,260 distinct negatives. These are Link-level negatives; the saved Skip-gram embeddings are unchanged.


In [5]:
drugs, diseases, positives = inputs()
assert len(drugs) == 1661 and len(diseases) == 840 and len(positives) == 5926
saved_seed0 = [(row['drug'], row['disease']) for row in cv.read_rows(SEED0_NEGATIVES)]
negative_sets = cv.sample_negative_sets(drugs, diseases, positives, saved_seed0, REPEAT_SEEDS)
assert negative_sets[0] == saved_seed0
all_negatives = [pair for seed in REPEAT_SEEDS for pair in negative_sets[seed]]
assert len(all_negatives) == len(set(all_negatives)) == 59260
assert not set(all_negatives).intersection(positives)
display(pd.DataFrame([
    dict(repeat_seed=seed, drugs=len(drugs), diseases=len(diseases),
         positives=len(positives), negatives=len(negative_sets[seed]),
         pairs=len(positives)+len(negative_sets[seed])) for seed in REPEAT_SEEDS]))


,repeat_seed,drugs,diseases,positives,negatives,pairs
0,0,1661,840,5926,5926,11852
1,1,1661,840,5926,5926,11852
2,2,1661,840,5926,5926,11852
3,3,1661,840,5926,5926,11852
4,4,1661,840,5926,5926,11852
5,5,1661,840,5926,5926,11852
6,6,1661,840,5926,5926,11852
7,7,1661,840,5926,5926,11852
8,8,1661,840,5926,5926,11852
9,9,1661,840,5926,5926,11852


## 4. Shared stratified folds and saved run identity

Each repeat uses sorted positives followed by its saved negative order. Every pair is tested once across ten folds; the other nine folds supply training data. Both classes are present in every training and test partition. The same saved assignments feed Original and Modified.

The run identity uses scientific inputs, settings and fold assignments. Notebook comments and Markdown are not checkpoint keys. Use a new `RUN_NAME` if scientific settings change; historical outputs are not overwritten.


In [6]:
repeats = {}
fold_sizes = []
for seed in REPEAT_SEEDS:
    pairs = cv.assign_folds(positives, negative_sets[seed], seed, N_FOLDS)
    assert len(pairs) == 11852
    assert {row['test_fold'] for row in pairs} == set(range(1, N_FOLDS+1))
    repeats[seed] = pairs
    for fold in range(1, N_FOLDS+1):
        selected = [row for row in pairs if row['test_fold'] == fold]
        assert {row['label'] for row in selected} == {0, 1}
        fold_sizes.append(dict(repeat_seed=seed, fold=fold, test_pairs=len(selected),
                               test_positives=sum(row['label'] for row in selected)))
display(pd.DataFrame(fold_sizes))


,repeat_seed,fold,test_pairs,test_positives
0,0,1,1186,593
1,0,2,1186,593
2,0,3,1185,593
3,0,4,1185,593
4,0,5,1185,593
...,...,...,...,...
95,9,6,1185,593
96,9,7,1185,592
97,9,8,1185,592
98,9,9,1185,592


In [7]:
input_paths = {
    'nodetypes': ROOT/'data/processed/msi_nodetypes.tsv',
    'positive_pairs': ROOT/'data/processed/msi_positive_pairs.tsv',
    'seed0_negatives': SEED0_NEGATIVES,
    'paper_reference': PAPER_REFERENCE,
    **{name+'_embedding': path for name, path in EMBEDDINGS.items()},
    **{name+'_'+filename: directory/filename
       for name, directory in HISTORICAL.items()
       for filename in ['run_config.json', 'fold_assignments.csv',
                        'pp1_fold_results.csv', 'pp1_evaluation_complete.json']},
}
config = dict(
    protocol_version=1, analysis='MSI Figure 2a: paired 10 x 10 CV with fixed embeddings',
    repeat_seeds=REPEAT_SEEDS, folds=N_FOLDS, embedding_seed=0, embeddings_fixed=True,
    tau=0.3, pp=1, mesh_year=2024, semantic_cutoff=0.4,
    classifier_seed_policy='repeat seed; identical across model variants',
    xgboost_base=XGB_PARAMETERS, feature='drug_embedding - disease_embedding',
    negative_sampling='global 1:1; uniform unknown pairs; preserve canonical seed0; disjoint across repeats',
    aggregation='mean of ten fold metrics per repeat; mean/sample SD/Student-t 95% CI across ten repeats',
    accuracy='numpy round of probability, matching historical evaluation', aupr='average_precision_score',
    cohort=dict(drugs=len(drugs), diseases=len(diseases), positives=len(positives)),
    inputs={name: dict(path=str(path.relative_to(ROOT)), sha256=sha256(path))
            for name, path in input_paths.items()},
    splits={str(seed): hashlib.sha256(json.dumps(pairs, sort_keys=True).encode()).hexdigest()
            for seed, pairs in repeats.items()},
    python=sys.version,
    packages={name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'scipy', 'xgboost']},
)
cv.open_run(RESULTS, config)
for seed in REPEAT_SEEDS:
    directory = RESULTS/'splits'/f'repeat{seed:02d}'
    cv.write_rows(directory/'negative_pairs.csv',
                  [dict(drug=drug, disease=disease) for drug, disease in negative_sets[seed]],
                  ['drug', 'disease'])
    cv.write_rows(directory/'fold_assignments.csv', repeats[seed], list(repeats[seed][0]))
cv.write_rows(RESULTS/'fold_sizes.csv', fold_sizes, list(fold_sizes[0]))
print('Saved shared evaluation inputs:', RESULTS)


Saved shared evaluation inputs: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_fixed_embeddings_10x10


## 5. Load the two existing embeddings

All required drugs and diseases must have finite 128D vectors. Each pair feature is the drug vector minus the disease vector. No scaler is fitted and no embeddings are updated during CV.


In [8]:
required_nodes = set(drugs) | set(diseases)
original_vectors = cv.load_vectors(EMBEDDINGS['original'], required_nodes)
modified_vectors = cv.load_vectors(EMBEDDINGS['modified'], required_nodes)
print('Embedding nodes:', len(original_vectors), 'original;', len(modified_vectors), 'modified')
example_x, example_y = cv.features(original_vectors, repeats[0][:2])
print('Example feature shape:', example_x.shape)


Embedding nodes: 29959 original; 29959 modified
Example feature shape: (2, 128)


## 6. Reuse the verified seed-0 ten-fold results

Import the existing metrics only after checking the embedding, ordered pair/fold IDs, XGBoost settings and completed result files. Each imported fold records its source files. Imported metrics do not create a new classifier or test-prediction file; their provenance remains the historical run.

The historical means are approximately AUROC 0.882394 for Original and 0.916308 for Modified. These imported results count as one repeat per model.


In [9]:
seed0_rows = []
for name in MODELS:
    rows = cv.import_seed0(HISTORICAL[name], EMBEDDINGS[name], repeats[0],
                           RESULTS/name/'repeat00', name, XGB_PARAMETERS, N_FOLDS)
    seed0_rows.extend(rows)
assert len(seed0_rows) == 20
print('Imported seed-0 fold means:')
display(pd.DataFrame(seed0_rows).groupby('model')[list(cv.METRICS)].mean())


Imported seed-0 fold means:


,accuracy,auroc,aupr
model,,,
modified,0.835220,0.916308,0.922352
original,0.802736,0.882394,0.889991


## 7. Original: seeds 1-9, ten folds per seed

This cell starts **90 new XGBoost fits**. The classifier seed equals the repeat seed. Each fold immediately saves `classifier.json`, `predictions.csv` and `metrics.json`; a later execution reuses completed folds.


In [10]:
original_rows = [row for row in seed0_rows if row['model'] == 'original']
for seed in REPEAT_SEEDS[1:]:
    pairs = repeats[seed]
    x, y = cv.features(original_vectors, pairs)
    parameters = dict(XGB_PARAMETERS, seed=seed)
    repeat_rows = []
    for fold in range(1, N_FOLDS+1):
        print(f'Original | seed {seed} | fold {fold}/{N_FOLDS}', flush=True)
        row = cv.fit_fold(x, y, pairs, 'original', seed, fold, parameters,
                          RESULTS/'original'/f'repeat{seed:02d}'/f'fold{fold:02d}')
        repeat_rows.append(row)
    original_rows.extend(repeat_rows)
    cv.write_rows(RESULTS/'original'/f'repeat{seed:02d}'/'fold_results.csv',
                  repeat_rows, list(repeat_rows[0]))
    display(pd.DataFrame(repeat_rows)[list(cv.METRICS)].mean().to_frame('fold_mean').T)
assert len(original_rows) == 100


Original | seed 1 | fold 1/10
Original | seed 1 | fold 2/10
Original | seed 1 | fold 3/10
Original | seed 1 | fold 4/10
Original | seed 1 | fold 5/10
Original | seed 1 | fold 6/10
Original | seed 1 | fold 7/10
Original | seed 1 | fold 8/10
Original | seed 1 | fold 9/10
Original | seed 1 | fold 10/10


,accuracy,auroc,aupr
fold_mean,0.801892,0.88574,0.895763


Original | seed 2 | fold 1/10
Original | seed 2 | fold 2/10
Original | seed 2 | fold 3/10
Original | seed 2 | fold 4/10
Original | seed 2 | fold 5/10
Original | seed 2 | fold 6/10
Original | seed 2 | fold 7/10
Original | seed 2 | fold 8/10
Original | seed 2 | fold 9/10
Original | seed 2 | fold 10/10


,accuracy,auroc,aupr
fold_mean,0.800538,0.883325,0.891231


Original | seed 3 | fold 1/10
Original | seed 3 | fold 2/10
Original | seed 3 | fold 3/10
Original | seed 3 | fold 4/10
Original | seed 3 | fold 5/10
Original | seed 3 | fold 6/10
Original | seed 3 | fold 7/10
Original | seed 3 | fold 8/10
Original | seed 3 | fold 9/10
Original | seed 3 | fold 10/10


...

,accuracy,auroc,aupr
fold_mean,0.804335,0.885936,0.895519


Original | seed 8 | fold 1/10
Original | seed 8 | fold 2/10
Original | seed 8 | fold 3/10
Original | seed 8 | fold 4/10
Original | seed 8 | fold 5/10
Original | seed 8 | fold 6/10
Original | seed 8 | fold 7/10
Original | seed 8 | fold 8/10
Original | seed 8 | fold 9/10
Original | seed 8 | fold 10/10


,accuracy,auroc,aupr
fold_mean,0.801891,0.884049,0.892645


Original | seed 9 | fold 1/10
Original | seed 9 | fold 2/10
Original | seed 9 | fold 3/10
Original | seed 9 | fold 4/10
Original | seed 9 | fold 5/10
Original | seed 9 | fold 6/10
Original | seed 9 | fold 7/10
Original | seed 9 | fold 8/10
Original | seed 9 | fold 9/10
Original | seed 9 | fold 10/10


,accuracy,auroc,aupr
fold_mean,0.802227,0.883003,0.89161


## 8. Modified: the identical seeds, negatives and folds

This cell starts **90 new fits** using the modified embeddings. Pair IDs, fold assignments, XGBoost parameters and classifier seeds match Section 7. Each completed fold is saved immediately.


In [11]:
modified_rows = [row for row in seed0_rows if row['model'] == 'modified']
for seed in REPEAT_SEEDS[1:]:
    pairs = repeats[seed]
    x, y = cv.features(modified_vectors, pairs)
    parameters = dict(XGB_PARAMETERS, seed=seed)
    repeat_rows = []
    for fold in range(1, N_FOLDS+1):
        print(f'Modified | seed {seed} | fold {fold}/{N_FOLDS}', flush=True)
        row = cv.fit_fold(x, y, pairs, 'modified', seed, fold, parameters,
                          RESULTS/'modified'/f'repeat{seed:02d}'/f'fold{fold:02d}')
        repeat_rows.append(row)
    modified_rows.extend(repeat_rows)
    cv.write_rows(RESULTS/'modified'/f'repeat{seed:02d}'/'fold_results.csv',
                  repeat_rows, list(repeat_rows[0]))
    display(pd.DataFrame(repeat_rows)[list(cv.METRICS)].mean().to_frame('fold_mean').T)
assert len(modified_rows) == 100


Modified | seed 1 | fold 1/10
Modified | seed 1 | fold 2/10
Modified | seed 1 | fold 3/10
Modified | seed 1 | fold 4/10
Modified | seed 1 | fold 5/10
Modified | seed 1 | fold 6/10
Modified | seed 1 | fold 7/10
Modified | seed 1 | fold 8/10
Modified | seed 1 | fold 9/10
Modified | seed 1 | fold 10/10


,accuracy,auroc,aupr
fold_mean,0.833363,0.916256,0.922112


Modified | seed 2 | fold 1/10
Modified | seed 2 | fold 2/10
Modified | seed 2 | fold 3/10
Modified | seed 2 | fold 4/10
Modified | seed 2 | fold 5/10
Modified | seed 2 | fold 6/10
Modified | seed 2 | fold 7/10
Modified | seed 2 | fold 8/10
Modified | seed 2 | fold 9/10
Modified | seed 2 | fold 10/10


,accuracy,auroc,aupr
fold_mean,0.834204,0.916244,0.92096


Modified | seed 3 | fold 1/10
Modified | seed 3 | fold 2/10
Modified | seed 3 | fold 3/10
Modified | seed 3 | fold 4/10
Modified | seed 3 | fold 5/10
Modified | seed 3 | fold 6/10
Modified | seed 3 | fold 7/10
Modified | seed 3 | fold 8/10
Modified | seed 3 | fold 9/10
Modified | seed 3 | fold 10/10


...

,accuracy,auroc,aupr
fold_mean,0.836566,0.916519,0.921042


Modified | seed 8 | fold 1/10
Modified | seed 8 | fold 2/10
Modified | seed 8 | fold 3/10
Modified | seed 8 | fold 4/10
Modified | seed 8 | fold 5/10
Modified | seed 8 | fold 6/10
Modified | seed 8 | fold 7/10
Modified | seed 8 | fold 8/10
Modified | seed 8 | fold 9/10
Modified | seed 8 | fold 10/10


,accuracy,auroc,aupr
fold_mean,0.834627,0.912028,0.916919


Modified | seed 9 | fold 1/10
Modified | seed 9 | fold 2/10
Modified | seed 9 | fold 3/10
Modified | seed 9 | fold 4/10
Modified | seed 9 | fold 5/10
Modified | seed 9 | fold 6/10
Modified | seed 9 | fold 7/10
Modified | seed 9 | fold 8/10
Modified | seed 9 | fold 9/10
Modified | seed 9 | fold 10/10


,accuracy,auroc,aupr
fold_mean,0.835216,0.913858,0.918985


## 9. Repeat means, uncertainty and paired improvement

First average each metric across the ten folds within a repeat. Then compute the mean, sample SD (`ddof=1`) and Student-t 95% CI across the ten repeat means. The 100 folds are not treated as 100 independent experiments.

Pair Original and Modified by repeat seed. Their differences use the same test pairs and folds. The intervals describe variation in Link-level negatives, folds and classifier training conditional on the two fixed embeddings; they exclude newly generated walks or embeddings.


In [12]:
all_rows = original_rows + modified_rows
repeat_rows, summary, differences, difference_summary = cv.summarize(all_rows, REPEAT_SEEDS, N_FOLDS)
for filename, rows in [
    ('fold_results.csv', all_rows), ('repeat_summary.csv', repeat_rows),
    ('summary.csv', summary), ('paired_repeat_deltas.csv', differences),
    ('paired_delta_summary.csv', difference_summary),
]:
    cv.write_rows(RESULTS/filename, rows, list(rows[0]))
print('Ten repeat means per model:')
display(pd.DataFrame(repeat_rows))
print('Mean, repeat-level SD and 95% CI:')
display(pd.DataFrame(summary))
print('Modified minus Original, paired by repeat:')
display(pd.DataFrame(differences))
display(pd.DataFrame(difference_summary))


Ten repeat means per model:


,model,repeat_seed,folds,accuracy,auroc,aupr
0,original,0,10,0.802736,0.882394,0.889991
1,original,1,10,0.801892,0.885740,0.895763
2,original,2,10,0.800538,0.883325,0.891231
3,original,3,10,0.801468,0.882189,0.889919
4,original,4,10,0.803748,0.886048,0.895645
5,original,5,10,0.800372,0.880987,0.886690
6,original,6,10,0.800453,0.882101,0.893882
7,original,7,10,0.804335,0.885936,0.895519
8,original,8,10,0.801891,0.884049,0.892645
9,original,9,10,0.802227,0.883003,0.891610


Mean, repeat-level SD and 95% CI:


,model,metric,mean,SD,CI95_low,CI95_high,repeats
0,modified,accuracy,0.834188,0.001291,0.833265,0.835112,10
1,modified,auroc,0.914619,0.001795,0.913335,0.915903,10
2,modified,aupr,0.919762,0.001965,0.918356,0.921168,10
3,original,accuracy,0.801966,0.001357,0.800995,0.802937,10
4,original,auroc,0.883577,0.001801,0.882289,0.884865,10
5,original,aupr,0.892289,0.002986,0.890153,0.894426,10


Modified minus Original, paired by repeat:


,model,repeat_seed,accuracy,auroc,aupr
0,modified_minus_original,0,0.032483,0.033913,0.032360
1,modified_minus_original,1,0.031471,0.030516,0.026349
2,modified_minus_original,2,0.033666,0.032919,0.029729
3,modified_minus_original,3,0.032400,0.030317,0.027651
4,modified_minus_original,4,0.028347,0.029892,0.025246
5,modified_minus_original,5,0.032484,0.032429,0.031015
6,modified_minus_original,6,0.033415,0.031012,0.025203
7,modified_minus_original,7,0.032231,0.030583,0.025523
8,modified_minus_original,8,0.032736,0.027979,0.024274
9,modified_minus_original,9,0.032989,0.030855,0.027374


,model,metric,mean,SD,CI95_low,CI95_high,repeats
0,modified_minus_original,accuracy,0.032222,0.001495,0.031153,0.033292,10
1,modified_minus_original,auroc,0.031042,0.001681,0.029839,0.032244,10
2,modified_minus_original,aupr,0.027473,0.002728,0.025521,0.029424,10


## 10. Compare with the author's Figure 2a Source Data

The reference CSV preserves the ten author DREAMwalk records with seeds 0-9. Its mean Accuracy/AUROC/AUPR is approximately **0.877098 / 0.944060 / 0.946767**. These are exact Source Data values, not figure estimates.

Local results use ten-fold means within each repeat. The workbook does not provide the authors' individual fold results. The local-to-paper comparison is descriptive: identical numeric seed labels do not recover their unpublished negative IDs or splits. Only the local Original-versus-Modified comparison has matched pairs and folds.

The paper's Figure 2 uses 95% CI error bars. The table below calculates Student-t intervals from the ten source rows and the ten local repeat means.


In [13]:
paper = pd.read_csv(PAPER_REFERENCE)
assert len(paper) == 10 and set(paper['repeat_seed']) == set(REPEAT_SEEDS)
paper_rows = [dict(model='paper_DREAMwalk', repeat_seed=int(row['repeat_seed']),
                   **{metric: float(row[metric]) for metric in cv.METRICS})
              for row in paper.to_dict('records')]
paper_summary = cv.mean_intervals(paper_rows)
reference = {row['metric']: row for row in paper_summary}
comparison = []
for row in summary:
    source = reference[row['metric']]
    comparison.append(dict(model=row['model'], metric=row['metric'],
                           local_mean=row['mean'], local_repeat_SD=row['SD'],
                           local_CI95_low=row['CI95_low'], local_CI95_high=row['CI95_high'],
                           paper_mean=source['mean'], paper_repeat_SD=source['SD'],
                           paper_CI95_low=source['CI95_low'], paper_CI95_high=source['CI95_high'],
                           local_minus_paper=row['mean']-source['mean']))
cv.write_rows(RESULTS/'paper_comparison.csv', comparison, list(comparison[0]))
print('Local Original, local Modified and published DREAMwalk:')
display(pd.DataFrame(summary + paper_summary))
display(pd.DataFrame(comparison))
cv.write_json(RESULTS/'evaluation_complete.json', dict(
    models=list(MODELS), repeat_seeds=REPEAT_SEEDS, folds_per_repeat=N_FOLDS,
    completed_fold_results=len(all_rows),
    imported_seed0_fold_results=sum(row['origin']=='historical_seed0' for row in all_rows),
    new_fold_results=sum(row['origin']=='new_fit' for row in all_rows),
    fixed_embeddings=True,
    uncertainty='repeat-level negative-set/fold/classifier variation conditional on fixed embeddings',
))
print('Completed evaluation:', RESULTS)


Local Original, local Modified and published DREAMwalk:


,model,metric,mean,SD,CI95_low,CI95_high,repeats
0,modified,accuracy,0.834188,0.001291,0.833265,0.835112,10
1,modified,auroc,0.914619,0.001795,0.913335,0.915903,10
2,modified,aupr,0.919762,0.001965,0.918356,0.921168,10
3,original,accuracy,0.801966,0.001357,0.800995,0.802937,10
4,original,auroc,0.883577,0.001801,0.882289,0.884865,10
5,original,aupr,0.892289,0.002986,0.890153,0.894426,10
6,paper_DREAMwalk,accuracy,0.877098,0.004838,0.873637,0.880559,10
7,paper_DREAMwalk,auroc,0.944060,0.004066,0.941152,0.946969,10
8,paper_DREAMwalk,aupr,0.946767,0.004258,0.943721,0.949813,10


,model,metric,local_mean,local_repeat_SD,local_CI95_low,local_CI95_high,paper_mean,paper_repeat_SD,paper_CI95_low,paper_CI95_high,local_minus_paper
0,modified,accuracy,0.834188,0.001291,0.833265,0.835112,0.877098,0.004838,0.873637,0.880559,-0.042910
1,modified,auroc,0.914619,0.001795,0.913335,0.915903,0.944060,0.004066,0.941152,0.946969,-0.029442
2,modified,aupr,0.919762,0.001965,0.918356,0.921168,0.946767,0.004258,0.943721,0.949813,-0.027005
3,original,accuracy,0.801966,0.001357,0.800995,0.802937,0.877098,0.004838,0.873637,0.880559,-0.075132
4,original,auroc,0.883577,0.001801,0.882289,0.884865,0.944060,0.004066,0.941152,0.946969,-0.060483
5,original,aupr,0.892289,0.002986,0.890153,0.894426,0.946767,0.004258,0.943721,0.949813,-0.054477


Completed evaluation: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_fixed_embeddings_10x10


## 11. Saved outputs and interpretation

All outputs are separate from historical runs, under `results/msi_mesh2024_fixed_embeddings_10x10/`:

- `splits/repeatXX/`: negative-pair IDs and common fold assignments.
- `original/` and `modified/`: imported seed-0 source records; per-fold classifiers, test predictions and completion records for seeds 1-9.
- `fold_results.csv`: all 200 fold results, with imported/new provenance.
- `repeat_summary.csv`: the 20 repeat-level means.
- `summary.csv`: model means, repeat SD and 95% CI.
- `paired_repeat_deltas.csv` and `paired_delta_summary.csv`: matched local improvement.
- `paper_comparison.csv`: comparison with the exact Figure 2a reference.

Read the repeated means alongside the existing seed-0 means. No hyperparameters are selected using test performance. This evaluation retains the reviewed MSI snapshot and fixed embeddings. It does not add disease-area splitting, model training seeds for new embeddings, or an HD case-study experiment.

Return to [the MSI experiment notebook](MSI_DREAMwalk_local.ipynb) for the earlier controls. Figure 2b remains in [its separate notebook](MSI_DREAMwalk_disease_area_split.ipynb).
